# Extract MFCC Features

Mel‑Frequency Cepstral Coefficients (MFCCs) are one of the most widely used features in audio and speech processing. They give you a compact, meaningful summary of the timbre or spectral shape of a sound.

For our use case, we will window our data and extract MFCCs for each window, and save them as features that will be used to train a classifier.

In [ ]:
!pip install librosa

We begin by loading the necessary packages and setting the necessary parameter variables.

In [ ]:
import numpy as np
import os
import helperFncs as hf
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as wav_write
import importlib
import librosa
from scipy import stats

# Directory path to the data
FOLDER_IN = '/home/jovyan/Data/Train/'
FOLDER_OUT_TRAIN = '/home/jovyan/Data/Post_MFCC/Train/'
FOLDER_OUT_VAL = '/home/jovyan/Data/Post_MFCC/Val/'

# Mapping of label indices to their corresponding class names
labelList = {0: 'Speech', 1: 'Cough', 2: 'Non-Verbal', 3: 'Other'}

# Specifying a window size, step size
windowSize = 1 # secconds
stepSize = 0.5 # seconds
nMFCC = 32 # Number of MFCC coefficients to extract

Splitting the files into training and validation.

In [ ]:
# Get a sorted list of all .h5 files in the input directory
fileList = sorted([f for f in os.listdir(FOLDER_IN) if f.endswith('.h5')])

# Splitting it into a training set and a validation set (last 6 files for validation).
# Assuming that they are in arbitrary order to begin with.
fileListTrain = fileList[:-6]
fileListVal = fileList[-6:]

Defining our main function for processing the data.

In [ ]:
def extractMFCC(fileList, folderOut):
    for fileName in fileList:
        # Load the selected .h5 file into a nested dictionary
        file_path = os.path.join(FOLDER_IN, fileName)
        data = hf.h5_to_dict(file_path)

        print(f"... Processing: {fileName}")

        # Extract relevant data from the loaded dictionary
        audioSamplingRate = data['audio_sample_rate'] # Sampling rate of the audio signal
        audioData = data['audio_data'] # Audio signal data
        labelSamplingRate = data['label_sample_rate'] # Sampling rate of the labels
        labels = data['label'] # Label data corresponding to the audio signal

        # Convert window/step from seconds to samples
        audio_win_len = int(windowSize * audioSamplingRate)
        audio_hop_len = int(stepSize * audioSamplingRate)
        label_win_len = int(windowSize * labelSamplingRate)
        label_hop_len = int(stepSize * labelSamplingRate)

        # Determine the number of windows we can extract from both audio and labels
        n_audio_windows = (len(audioData) - audio_win_len) // audio_hop_len + 1
        n_label_windows = (len(labels) - label_win_len) // label_hop_len + 1
        n_windows = min(n_audio_windows, n_label_windows)

        mfcc_samples = []
        label_samples = []
        middle_time = []
        data_mfcc = {}

        # Slide over audioData and labels together and compute MFCC per window
        for i in range(n_windows):
            audio_start = i * audio_hop_len
            label_start = i * label_hop_len

            segment_audio = audioData[audio_start:audio_start + audio_win_len]
            segment_label = labels[label_start:label_start + label_win_len]

            mfcc = librosa.feature.mfcc(
                y=segment_audio.astype(float),
                sr=audioSamplingRate,
                n_mfcc=nMFCC,
                n_fft=audio_win_len,
                hop_length=audio_hop_len,
                center=False
            )
            # Store the MFCC for this segment
            mfcc_samples.append(mfcc)

            # Store the mode of this segment
            mode_label = stats.mode(segment_label, axis=None)
            mode_label = mode_label.mode
            label_samples.append(mode_label)
            
            # Store the middle time of this window in seconds
            middle_time.append((audio_start + audio_win_len/2.0)/audioSamplingRate)

        # Convert lists to numpy arrays and save to a new .h5 file
        data_mfcc['mfcc'] = np.stack(mfcc_samples, axis=0).squeeze(-1)
        data_mfcc['labels'] = np.array(label_samples)
        data_mfcc['middle_time'] = np.array(middle_time)

        outPath = os.path.join(folderOut, os.path.splitext(fileName)[0] + '.h5')
        hf.write_h5(outPath, data_mfcc)
        print(f"... Saved MFCC data to: {outPath}")

Processing and storing the training and validation sets. 

In [ ]:
# Extract MFCC features for both training and validation sets
print('Processing Training Set')
extractMFCC(fileListTrain, FOLDER_OUT_TRAIN)
print('Processing Validation Set')
extractMFCC(fileListVal, FOLDER_OUT_VAL)